In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter
import json

import preprocessing as pp
import candidates as ca
import refinements as rf
import subgroup_selection as ss
import search as bs

In [2]:
unclean_data = pd.read_csv('algebrakit_firstbatch_2.csv')
df = pp.clean_data_algebrakit(unfilterd_data=unclean_data, filter_type="remove_interaction")
print(f"we have {len(df)} records of {len(df['sessionID'].unique())} sessions with {df['finished'].value_counts()}")

we have 6020 records of 1622 sessions with finished
False    3958
True     2062
Name: count, dtype: int64


In [24]:
df

,exerciseID,sessionID,success,creationTimestamp,skills,stepTypes,finished,marksEarned,marksTotal,questionID,...,result_progress,marksEarned_interaction,marksTotal_interaction,finished_interaction,maxAttempts,usedAttempts,failed,marksPenalty,hintsRequested,mathErrors
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.00,4.0,Q1,...,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,[FractionArithmeticMultiplicationProper],[CALCULATE],False,1.00,4.0,Q2,...,0.0,0.00,1.0,False,2.0,2.0,True,0.00,1,2
2,9f4a675e-c2b3-4e11-91cc-4445951bab60,03bba4c5-afea-42c0-87bb-f99e5f2d2bfb,True,1744026958,[FractionArithmeticMultiplicationProper],[CALCULATE],True,0.65,4.0,Q1,...,1.0,0.65,1.0,True,2.0,1.0,False,0.35,1,1
3,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,True,1741946748,[FractionArithmeticMultiplicationProper],[CALCULATE],True,4.00,4.0,Q1,...,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0
4,9f4a675e-c2b3-4e11-91cc-4445951bab60,040b59bb-b5da-4ef0-ba5b-3e23f163c7de,True,1741946748,[FractionArithmeticMultiplicationProper],[CALCULATE],True,4.00,4.0,Q2,...,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6015,0beaa764-f558-4124-8d4d-b224ebdb2d4a,f5ad2de1-c604-4326-8c8a-d13ff9726879,True,1725882783,"[EquationsBalanceMethodArrangeTerms, Equations...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]",False,3.65,6.0,Q8,...,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0
6016,0beaa764-f558-4124-8d4d-b224ebdb2d4a,f74e86aa-d869-4507-b561-501309ea5a37,True,1725387964,"[EquationsBalanceMethodArrangeTerms, Equations...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]",True,3.00,6.0,Q5,...,0.0,0.00,1.0,False,NaN,NaN,NaN,0.50,0,1
6017,0beaa764-f558-4124-8d4d-b224ebdb2d4a,f74e86aa-d869-4507-b561-501309ea5a37,True,1725387964,"[EquationsBalanceMethodArrangeTerms, Equations...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]",True,3.00,6.0,Q6,...,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0
6018,0beaa764-f558-4124-8d4d-b224ebdb2d4a,f74e86aa-d869-4507-b561-501309ea5a37,True,1725387964,"[EquationsBalanceMethodArrangeTerms, Equations...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]",True,3.00,6.0,Q7,...,1.0,1.00,1.0,True,2.0,0.0,False,0.00,0,0


In [15]:
df_skills = df[['skills']]
type( df_skills.iloc[0]['skills'] )

list

In [ ]:
# Step 2: One-hot encoding with empty list handling
one_hot = pd.get_dummies(df_skills['skills'].apply(pd.Series).stack()).groupby(level=0).sum()

# Ensure rows with empty lists are handled
df_one_hot = df_skills.drop('skills', axis=1).join(one_hot, how='left').fillna(0).astype(bool)

df_one_hot

,AlgebraAddingLikeTerms,AlgebraProductOfPowerFunctions,ArithmeticOrderDivision,ArithmeticOrderMultiplication,EquationsBalanceMethodArrangeTerms,EquationsBalanceMethodDivideCoefficient,EquationsOperationsAdd,EquationsOperationsMultiply,ExpandSingleBrackets,FactorSingleBrackets,...,FractionArithmeticSimilarImproperToMixed,FractionArithmeticSimilarMixed,FractionArithmeticSimilarMixedToImproper,FractionArithmeticSimplifyProper,NumberArithmeticAddConcept,NumberArithmeticAdditionDecimal,NumberArithmeticAdditionInteger,NumberArithmeticMultiplicationDecimal,NumberArithmeticMultiplicationInteger,NumberArithmeticPower
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6015,True,False,False,False,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False
6016,True,False,False,False,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False
6017,True,False,False,False,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False
6018,True,False,False,False,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,False


In [23]:
for i in df_skills[6015:]['skills']:
    print(i)

['EquationsBalanceMethodArrangeTerms', 'EquationsOperationsAdd', 'AlgebraAddingLikeTerms', 'EquationsBalanceMethodDivideCoefficient', 'EquationsOperationsMultiply']
['EquationsBalanceMethodArrangeTerms', 'EquationsOperationsAdd', 'AlgebraAddingLikeTerms', 'EquationsBalanceMethodDivideCoefficient', 'EquationsOperationsMultiply']
['EquationsBalanceMethodArrangeTerms', 'EquationsOperationsAdd', 'AlgebraAddingLikeTerms', 'EquationsBalanceMethodDivideCoefficient', 'EquationsOperationsMultiply']
['EquationsBalanceMethodArrangeTerms', 'EquationsOperationsAdd', 'AlgebraAddingLikeTerms', 'EquationsBalanceMethodDivideCoefficient', 'EquationsOperationsMultiply']
['EquationsBalanceMethodArrangeTerms', 'EquationsOperationsAdd', 'AlgebraAddingLikeTerms', 'EquationsBalanceMethodDivideCoefficient', 'EquationsOperationsMultiply']
